# Notebook 4 — The Real-Time Audit: Fresh-Context Evaluation

**Goal:** measure whether the model actually *knows* today's news, by asking
the questions generated in Notebook 3 in a completely fresh context window —
**without** the article and **without** web search.

## Context leakage: the audit-killer

If the evaluated model can see the scraped article — in the prompt, in shared
conversation history, or via the same session that generated the questions —
the audit collapses into a **reading-comprehension test**, and near-perfect
scores tell you nothing about real-time knowledge. Leakage is subtle:

- reusing the generator's chat session (history carries the article),
- system prompts that accumulate "helpful context",
- retrieval middleware silently injecting the source document.

The design rule: **one independent, stateless API request per question**,
containing only the question and its four options.

## Why no web search here (unlike Notebook 1)?

Notebook 1's open-book condition measured *retrieval-augmented* accuracy. This
audit deliberately measures the opposite: the model's **parametric real-time
awareness** — what it knows with no tools at all. Every call therefore uses
`use_web_search=False`. (Rerunning this notebook with `use_web_search=True`
is a great exercise: it converts the audit into a measure of how well live
search recovers today's facts.)

## Interpreting the score

- **~25%** is the random-guess floor for 4-option items.
- Scores near the floor on today's news are *expected* for a tool-less model
  with a training cutoff — that is the point of the exercise.
- Scores well above the floor invite scrutiny: is the "news" actually old
  (fallback samples, evergreen facts), or are distractors weak enough to
  eliminate by common sense? Read the justifications to find out.


In [ ]:
import os
from pathlib import Path

# Notebooks live in notebooks/; run from the repo root so relative paths like
# ./data behave identically in VS Code and classic Jupyter. The local
# `toolkit` package is installed (editable) in the .venv, so no sys.path hacks.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

import json

import pandas as pd
from pydantic import BaseModel, Field

from toolkit.config import DEFAULT_LLM, QUESTIONS_DIR
from toolkit.providers import openai_provider

question_files = sorted(Path(QUESTIONS_DIR).glob("question_*.json"))
if not question_files:
    raise FileNotFoundError(
        "No question files in data/questions/ — run Notebook 3 first."
    )

audit_items = []
for path in question_files:
    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    for q in payload["questions"]:
        audit_items.append({"source_article": payload["source_article"], **q})

print(f"Loaded {len(audit_items)} questions from {len(question_files)} files")


## 1. Evaluation schema and the stateless asker

Note what each request does **not** contain: no article, no hints, no shared
history, no tools. Each call constructs a brand-new message list.


In [ ]:
class EvaluationVerdict(BaseModel):
    chosen_option_index: int = Field(description="The index of the selected option.")
    justification: str = Field(description="Reasoning explaining why this is the correct choice.")


EVALUATOR_SYSTEM = (
    "You are answering a multiple-choice quiz about current events. Answer "
    "from your own knowledge only. If you are not sure, choose the most "
    "plausible option — and say in your justification that you are unsure."
)


def answer_question(question, options):
    option_block = "\n".join(f"{i}. {opt}" for i, opt in enumerate(options))
    parsed, _raw = openai_provider.run_parsed(
        DEFAULT_LLM,
        EVALUATOR_SYSTEM,
        f"{question}\n\nOptions:\n{option_block}\n\nPick the index of the correct option.",
        use_web_search=False,  # parametric awareness only — no tools
        text_format=EvaluationVerdict,
    )
    return parsed


In [ ]:
rows = []
for item in audit_items:
    try:
        verdict = answer_question(item["question"], item["options"])
        chosen = verdict.chosen_option_index
        in_range = 0 <= chosen < len(item["options"])
        correct = in_range and chosen == item["correct_index"]
        rows.append({
            "source_article": item["source_article"],
            "question": item["question"],
            "correct_index": item["correct_index"],
            "chosen_index": chosen if in_range else None,
            "correct": correct,
            "answered": True,
            "justification": verdict.justification,
        })
        status = "correct" if correct else "WRONG"
    except Exception as e:
        rows.append({
            "source_article": item["source_article"],
            "question": item["question"],
            "correct_index": item["correct_index"],
            "chosen_index": None,
            "correct": False,
            "answered": False,
            "justification": f"API error: {e}",
        })
        status = "ERROR"
    print(f"[{status:^7}] {item['question'][:70]}...")

report_df = pd.DataFrame(rows)
report_df.to_csv("data/nb04_report.csv", index=False)
print("\nSaved -> data/nb04_report.csv")


## 2. Report


In [ ]:
answered = report_df[report_df["answered"]]
n_answered, n_total = len(answered), len(report_df)
accuracy = answered["correct"].mean() if n_answered else float("nan")

print("REAL-TIME AUDIT REPORT")
print("=" * 50)
print(f"Questions asked      : {n_total}")
print(f"Answered (no errors) : {n_answered}")
print(f"Overall accuracy     : {accuracy:.1%}")
print(f"Random-guess floor   : 25.0%")
print(f"Lift over guessing   : {accuracy - 0.25:+.1%}")
print()
print("NOTE: with only ~9 items this is a demonstration, not an estimate —")
print("      a real audit would run hundreds of items across many days.")

answered.groupby("source_article").agg(
    accuracy=("correct", "mean"), n=("correct", "size")
).round(3)


In [ ]:
# Qualitative pass: read the justifications on failures. Did the model admit
# uncertainty (honest ignorance) or confidently assert something false?
failures = answered[~answered["correct"]]
for row in failures.itertuples():
    print("-" * 78)
    print(f"Q: {row.question}")
    print(f"   chose option {row.chosen_index}, correct was {row.correct_index}")
    print(f"   justification: {row.justification[:300]}")


## Takeaways

- The fresh-context, tool-less design measures **parametric real-time
  awareness** — the same quantity a user gets when they ask the model about
  the news with no retrieval attached.
- Accuracy near the guess floor is the *expected honest result* on genuinely
  new events; the interesting signal is in the justifications — models that
  confidently narrate wrong answers are more dangerous than models that
  guess and say so.
- Results vary by run day; that variance is a feature — rerun the pipeline
  (Notebooks 2 → 3 → 4) on a different day and compare.

**Next:** Notebook 5 replaces the single judge with a multi-agent debate
among heterogeneous models.
